In [1]:
from datasets import load_dataset
import torch
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import os
import re

In [2]:
# Loading the tmdb dataset
ds = load_dataset("ada-datadruids/full_tmdb_movies_dataset")

# Loading the mpent model for embeddings
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

TMDB_movie_dataset_v11.csv:   0%|          | 0.00/524M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1142342 [00:00<?, ? examples/s]

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords'],
        num_rows: 1142342
    })
})

In [11]:
movie = ds['train'][0]
f'Overview: {movie["title"]}, {movie["overview"]}, {movie["tagline"]}. Keywords: {movie["keywords"]}'

'Overview: Inception, Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person\'s idea into a target\'s subconscious., Your mind is the scene of the crime.. Keywords: rescue, mission, dream, airplane, paris, france, virtual reality, kidnapping, philosophy, spy, allegory, manipulation, car crash, heist, memory, architecture, los angeles, california, dream world, subconscious'

In [16]:
def combine_columns(movie):
    "A function to create the input text for training"
    embedding = f'Overview: {movie["title"]}, {movie["overview"]}, {movie["tagline"]}. Keywords: {movie["keywords"]}'

    return {'embedding': embedding}

# Applying the function to the top 100,000 movie only
first_100_000 = ds['train'].select(range(100_000))
first_100_000 = first_100_000.map(combine_columns)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [31]:
# Saving the embeddings after each batch
batch_size = len(first_100_000) // 10
folder_path = "./tmdb embeddings/"
start = 0
for batch in tqdm(range(10)):
    end = start + batch_size
    movies_embeddings = model.encode(first_100_000['embedding'][start:end-1])
    filename = f"embeddings_{start}-{end}.npy"
    save_path = os.path.join(folder_path, filename)
    np.save(save_path, movies_embeddings)
    start = end

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [02:22<00:00, 14.26s/it]
